# **비지도학습(Unsupervised Learning)**



---



**(코랩에서)한글 폰트 지정하는 방법**

In [ ]:
# 코랩에서 실행
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import shutil

# ── 1. 나눔 폰트 설치 ──────────────────────────────────────────
!apt-get install -y fonts-nanum > /dev/null 2>&1

# ── 2. matplotlib 캐시 디렉터리 삭제 ──────────────────────────
shutil.rmtree(mpl.get_cachedir(), ignore_errors=True)

# ── 3. 폰트 등록  ───────────────────────────────────
font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (6, 4)

- 사용 라이브러리

In [ ]:
import os, time, zipfile, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
SEED = 42

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, HDBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

- 사용 데이터셋

In [ ]:
def first_existing(*paths):
    return next((p for p in paths if os.path.exists(p)), paths[0])

FRUITS = first_existing("fruits_300.npy", "fruits_300.npy")
CARD_ZIP = first_existing("VL_카드거래.zip", "VL_카드거래.zip")
fruits = np.load(FRUITS)                                     # (300, 100, 100) · 사과/파인애플/바나나 각 100장
fruits_2d = fruits.reshape(-1, 100 * 100).astype(float)      # 이미지 1장 = 10,000차원 벡터
fruit_true = np.repeat([0, 1, 2], 100)                       # 정답(평가용으로만 사용! 학습에는 쓰지 않는다)
print("fruits:", fruits.shape, "| card zip 존재:", os.path.exists(CARD_ZIP))



---



## **비지도학습이란?**

- **레이블(정답)이 없는 데이터에서 패턴, 구조, 관계를 스스로 찾아내는 머신러닝 방법**
- **현실 세계의 대부분 데이터는 레이블이 없음.**
    - ex) 유튜브에 업로드 되는 수억 개의 영상에 일일이 사람이 분류 태그를 달기는 불가능함. 이처럼 레이블링 비용이 너무 크거나, 데이터의 숨겨진 패턴을 발견하고 싶을 때 비지도학습이 필수적임
- 주로 데이터를 비슷한 특성을 가진 그룹으로 묶는 **군집화(Clustering)**,
- 데이터의 복잡성을 줄이는 **차원 축소(Dimensionality Reduction)**,
- 데이터 간의 연관성을 파악하는 **연관 규칙 학습(Association Rule Learning)** 등에 사용됨

## **언제 비지도학습을 사용해야 할까?**

- **적합한 상황**
    - 레이블링 비용이 너무 높을 때
    - 데이터의 숨겨진 패턴을 탐색하고 싶을 때
    - 데이터 전처리 단계 (차원 축소, 이상치 제거)
    - 지도학습의 사전 단계 (특징 추출)
- **부적합한 상황**
    - 명확한 예측 목표가 있고 충분한 레이블 데이터가 있을 때
    - 실시간 정확도가 중요한 의료 진단, 금융 거래 등

### **[실습] 과일사진 자동 분류하기**

#### **1. 과일 사진 데이터 준비하기**

In [ ]:
# 웹에서 데이터 다운로드하기
!wget https://bit.ly/fruits_300_data -O fruits_300.npy

In [ ]:
# 데이터 로드하기
import numpy as np
import matplotlib.pyplot as plt

fruits = np.load('fruits_300.npy')

print('✅ dataset shape', fruits.shape)
# (300, 100, 100) :  300개의 100x100 사이즈 데이터

# 첫 번째 행 데이터 출력하기
print('✅ 첫 번째 행 데이터 & 시각화: ')
print(fruits[0, 0, :])
plt.imshow(fruits[0], cmap='gray')
plt.show()

# 우리가 관심있는 것이 과일이란 것을 강조하기 위해 바탕색과 과일색 반전해서 보기(gray_r)
print('\n✅ 바탕색과 과일색 반전해서 보기(gray_r)')
plt.imshow(fruits[0], cmap='gray_r')
plt.show()


# 50번, 150번, 250번째 데이터 시각화하기
print('\n✅ 50번, 150번, 250번째 데이터 시각화')
fig, axs = plt.subplots(1, 3)
axs[0].imshow(fruits[50], cmap='gray_r')
axs[1].imshow(fruits[150], cmap='gray_r')
axs[2].imshow(fruits[250], cmap='gray_r')
plt.show()

#### **2. 픽셀 값 분석하기**

In [ ]:
# 각 과일별로 데이터를 1차원 배열(100개의 10,000:100x100)로 만들기
apple = fruits[0:100].reshape(-1, 100*100)
pineapple = fruits[100:200].reshape(-1, 100*100)
banana = fruits[200:300].reshape(-1, 100*100)

print(apple.shape)          # apple 데이터의 모양
print(apple.mean(axis=1))   # apple 데이터들의 평균

In [ ]:
# 각 샘플 데이터의 평균을 히스토그램을 시각화
plt.hist(np.mean(apple, axis=1), alpha=0.8)
plt.hist(np.mean(pineapple, axis=1), alpha=0.8)
plt.hist(np.mean(banana, axis=1), alpha=0.8)
plt.legend(['apple', 'pineapple', 'banana'])
plt.show()

In [ ]:
# 전체 샘플에 대한 픽셀의 평균 시각화
fig, axs = plt.subplots(1, 3, figsize=(20, 5))
axs[0].bar(range(10000), np.mean(apple, axis=0))
axs[1].bar(range(10000), np.mean(pineapple, axis=0))
axs[2].bar(range(10000), np.mean(banana, axis=0))
plt.show()

In [ ]:
# 100x100 크기로 바꿔서 픽셀의 평균값 계산
apple_mean = np.mean(apple, axis=0).reshape(100, 100)
pineapple_mean = np.mean(pineapple, axis=0).reshape(100, 100)
banana_mean = np.mean(banana, axis=0).reshape(100, 100)

# 픽셀 평균값을 시각화 --> 과일별 대표 이미지
fig, axs = plt.subplots(1, 3, figsize=(20, 5))
axs[0].imshow(apple_mean, cmap='gray_r')
axs[1].imshow(pineapple_mean, cmap='gray_r')
axs[2].imshow(banana_mean, cmap='gray_r')
plt.show()

**3. 평균값과 가까운 사진 고르기**

In [ ]:
# 절대값 오차의 평균
abs_diff = np.abs(fruits - apple_mean)
abs_mean = np.mean(abs_diff, axis=(1,2))
print(abs_mean.shape)

In [ ]:
# 절대값 오차 평균 값이 작은수 100개 시각화
apple_index = np.argsort(abs_mean)[:100]
fig, axs = plt.subplots(10, 10, figsize=(10,10))
for i in range(10):
    for j in range(10):
        axs[i, j].imshow(fruits[apple_index[i*10 + j]], cmap='gray_r')
        axs[i, j].axis('off')
plt.show()

- **[확인문제]**
    - 앞에서 사과의 평균값과 가까운 사진 고르기 방법을 이용하여 바나나의 평균값과 가까운 사진을 골라 시각화 해보세요.

In [ ]:
abs_diff = np.abs(fruits - banana_mean)
abs_mean = np.mean(abs_diff, axis=(1,2))

banana_index = np.argsort(abs_mean)[:100]
fig, axs = plt.subplots(10, 10, figsize=(10,10))
for i in range(10):
    for j in range(10):
        axs[i, j].imshow(fruits[banana_index[i*10 + j]], cmap='gray_r')
        axs[i, j].axis('off')
plt.show()

### [LLM 실습] LLM에게 데이터 묶어보게 하기

In [ ]:
!pip install -q google-genai

In [ ]:
import numpy as np
from google import genai
from google.colab import userdata

# Gemini API 연결
client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

# ------------------------------------------------------------
# 1. 분류할 음식 데이터
# ------------------------------------------------------------
items = ["김치찌개", "파스타", "된장국", "피자", "비빔밥", "리조또", "불고기", "라자냐", "떡볶이", "카르보나라"]
items = [
    # 한식
    "김치찌개", "된장국", "비빔밥", "불고기", "떡볶이",
    "삼계탕", "잡채", "갈비탕", "냉면", "김밥",

    # 이탈리아 음식
    "파스타", "피자", "리조또", "라자냐", "카르보나라",
    "뇨키", "라비올리", "미네스트로네", "티라미수", "브루스케타",

    # 일본 음식
    "초밥", "라멘", "우동", "돈카츠", "오코노미야키",

    # 중국 음식
    "마파두부", "짜장면", "탕수육", "딤섬", "볶음밥"
]

# 리스트 섞기
np.random.shuffle(items)

# ------------------------------------------------------------
# 2. 자연스러운 그룹 만들기
#    temperature를 낮게 설정 → 결과의 일관성 증가
# ------------------------------------------------------------
TEMPERATURE = 0.2

r = client.interactions.create(
    model="gemini-3-flash-preview",

    input=f"""
아래 항목을 미리 정해진 정답 없이 자연스러운 그룹으로 나눠라.

조건:
1. 그룹 개수도 스스로 결정한다.
2. 각 그룹에 적절한 이름을 붙인다.
3. 각 그룹을 그렇게 나눈 이유를 한 줄로 설명한다.
4. 모든 음식은 반드시 하나의 그룹에 포함한다.

음식 목록:
{items}
""",

    generation_config={
        "temperature": TEMPERATURE
    }
)

print("===== 그룹 분류 결과 =====")
print(r.output_text)

# ------------------------------------------------------------
# 3. 같은 요청을 여러 번 반복
#    그룹 개수가 얼마나 일관적인지 확인
# ------------------------------------------------------------
print("\n===== 반복 실험 =====")

for i in range(5):
    out = client.interactions.create(
        model="gemini-3-flash-preview",

        input=f"""
다음 음식들을 의미적으로 자연스러운 그룹으로 나눠라.

{items}

다른 설명은 하지 말고
최종적으로 만들어진 '그룹의 개수'만 숫자로 출력하라.
""",

        generation_config={
            "temperature": TEMPERATURE
        }

    ).output_text

    print(f"{i+1}회 실행 : {out.strip()}")



---



# **군집화(Clustering)**

- K-Means : “가까운 중심점을 찾아가자.”
- DBSCAN : “주변에 이웃이 충분히 많으면 같은 동네다.”
- HDBSCAN : “동네마다 인구 밀도가 달라도, 밀도 구조를 계층적으로 찾아보자.”

## **1. K-평균 군집화**

- K-평균 군집화 알고리즘
    - K개의 군집을 나누기 위해 군집의 중심(Centroid)과 데이터 간의 평균 거리를 활용함
    - 각 군집 내 분산을 최소화하는 것을 목적으로 군집화 진행
    - 계산량이 적기 때문에 대용량 데이터도 빠르게 처리할 수 있는 장점이 있다
    
| 연도 | 알고리즘 | 아이디어 |
| --- | --- | --- |
| 1957/1982 · 1967 | K-means (Lloyd; MacQueen) | 중심까지 거리 제곱합 최소화 |
| 1977 | EM 알고리즘 (Dempster 외) → GMM | 확률 분포의 혼합, 소프트 군집 |
| 1996 | DBSCAN (Ester 외) | 밀도가 높은 곳을 잇는다, 잡음 분리 |
| 2007 | k-means++ (Arthur & Vassilvitskii) | 초기 중심을 멀리 퍼뜨려 뽑기 |
| 2013 | HDBSCAN (Campello 외) | 여러 밀도를 한 번에, eps 불필요 |

### [실습] K-means를 Numpy로 직접 구현
목적함수 (WCSS, inertia): $J=\sum_{k=1}^{K}\sum_{x_i\in C_k}\lVert x_i-\mu_k\rVert^2$
1. **배정 단계**: 각 점을 가장 가까운 중심에 배정 → J 감소 (또는 유지)
2. **갱신 단계**: 중심을 군집 평균으로 이동 → J 감소 (평균이 거리 제곱합을 최소로 하는 점이므로)
→ J는 매 반복마다 줄어들고 유한한 배정 경우의 수 때문에 **반드시 수렴**한다(단, 전역 최적 보장은 없음)

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.datasets import make_blobs

SEED = 42
X, y_true = make_blobs(n_samples=600,
                       centers=4, random_state=SEED)


def kmeans_numpy(X, k, n_iter=100, seed=0, plus_plus=True):
    rng = np.random.default_rng(seed)
    if plus_plus:                                              # k-means++ 초기화
        C = [X[rng.integers(len(X))]]
        for _ in range(k - 1):
            d2 = np.min(((X[:, None] - np.array(C)[None]) ** 2).sum(-1), axis=1)
            C.append(X[rng.choice(len(X), p=d2 / d2.sum())])  # 먼 점일수록 뽑힐 확률 ↑
        C = np.array(C)
    else:
        C = X[rng.choice(len(X), k, replace=False)]
    history = []
    for _ in range(n_iter):
        D = ((X[:, None, :] - C[None, :, :]) ** 2).sum(-1)    # (n, k) 거리 제곱
        labels = D.argmin(1)                                   # ① 배정
        history.append(D[np.arange(len(X)), labels].sum())    # 목적함수 J 기록
        newC = np.array([X[labels == j].mean(0) if (labels == j).any() else C[j] for j in range(k)])  # ② 갱신
        if np.allclose(newC, C):
            break
        C = newC
    return labels, C, history


labels, C, hist = kmeans_numpy(X, 4)
sk = KMeans(4, n_init=10, random_state=SEED).fit(X)
print("✅ 반복별 J:", np.round(hist, 1))
print(f"✅ 직접 구현 J={hist[-1]:.1f}  vs  sklearn inertia_={sk.inertia_:.1f}  | ARI(직접 vs 정답)={adjusted_rand_score(y_true, labels):.3f}")

# 초기화 비교: 무작위 초기화 20번 vs k-means++ 20번 — 나쁜 지역 최적에 빠지는 비율
bad = lambda pp: np.mean([kmeans_numpy(X, 4, seed=s, plus_plus=pp)[2][-1] > sk.inertia_ * 1.05 for s in range(20)])
print(f"✅ 나쁜 해(J가 최적보다 5%↑) 비율 — 무작위 초기화 {bad(False):.0%} · k-means++ {bad(True):.0%}")

### **[실습] K-Means를 사용하여 과일 사진 분류하기**  

**1. 데이터 준비**

In [ ]:
# 필요한 라이브러리를 불러옵니다.
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# 1.fruits_300.npy 파일 로드하기
# (300, 100, 100) : 100x100 크기의 이미지 300장.
fruits = np.load('fruits_300.npy')

# 2. 사이킷런 머신러닝 모델에 사용하기 위해 3차원 배열을 2차원 배열로 변경
# (300, 100, 100) -> (300, 10000): 각 이미지를 10000개의 픽셀 값을 가진 하나의 샘플로 만듦
fruits_2d = fruits.reshape(-1, 100*100)


**2. K-평균 모델 훈련**

In [ ]:
# 3. K-평균 모델을 생성하고 훈련시킴
# n_clusters=3 : 데이터를 3개의 그룹으로 나누도록 설정. (사과, 바나나, 파인애플)
km = KMeans(n_clusters=3, random_state=42)
km.fit(fruits_2d)

# 4. 각 이미지(샘플)가 어떤 클러스터에 속하는지 확인.
# km.labels_ : 0, 1, 2 중 하나의 값을 가진 300개의 원소로 이루어진 배열
print("✅ 각 샘플의 클러스터 레이블:")
print(km.labels_)
print("✅ 샘플의 고유 클러스터 레이블:")
print(np.unique(km.labels_, return_counts=True))

**3. 클러스터링 결과 시각화**

In [ ]:
def draw_fruits(arr, ratio=1):
    n = len(arr)    # n은 샘플 개수입니다

    # 한 줄에 10개씩 이미지를 그린다. 샘플 개수를 10으로 나누어 전체 행 개수를 계산
    rows = int(np.ceil(n/10))

    # 행이 1개 면 열 개수는 샘플 개수, 그렇지 않으면 10개.
    cols = n if rows < 2 else 10
    print(f'rows:{rows}, cols:{cols}')

    fig, axs = plt.subplots(rows, cols,
                            figsize=(cols*ratio, rows*ratio), squeeze=False)
    for i in range(rows):
        for j in range(cols):
            if i*10 + j < n:    # n 개까지만 그린다.
                axs[i, j].imshow(arr[i*10 + j], cmap='gray_r')
            axs[i, j].axis('off')
    plt.show()

draw_fruits(fruits[km.labels_==0])

In [ ]:
draw_fruits(fruits[km.labels_==1])

In [ ]:
draw_fruits(fruits[km.labels_==2])

**4. 클러스터 중심 확인하기**

In [ ]:
# fruits_2d 샘플의 클러스터 중심을 이미지로 확인하기
draw_fruits(km.cluster_centers_.reshape(-1, 100, 100), ratio=3)

In [ ]:
# transform() : 훈련 데이터 샘플에서 클러스터 중심까지 거리로 변환해주는 함수
# 인덱스가 100인 샘플 --> 가장 작은 값으로 분류됨
print(km.transform(fruits_2d[100:101]))

# 앞의 분류 결과가 맞는지 확인
print(km.predict(fruits_2d[100:101]))

# 그림으로도 확인
draw_fruits(fruits[100:101])

In [ ]:
# k-means 알고리즘은 반복적으로 클러스터 중심을 옮기면서 최적의 클러스터를 찾는다.
# 알고리즘의 반복한 횟수 확인: (n_iter_ 속성에 저장됨)
print(km.n_iter_)

### **최적의 k 찾기**

- 군집 알고리즘에서 적절한 K값을 찾기 위한 완벽한 방법은 없다.
- K는 몇 개? — 엘보(inertia) : 대표적인 방법
- 실루엣 계수
    - 실루엣: $s(i)=\dfrac{b(i)-a(i)}{\max(a(i),b(i))}$  
        - a: 같은 군집 평균거리,
        - b: 가장 가까운 다른 군집 평균거리)
        - -1 ~ 1, 클수록 좋음
    - 클러스터 개수를 늘려가면서 이너셔의 변화를 관찰하여 최적의 클러스터 개수를 찾는 방법
    - **이너셔(inertia)** : 클러스터 중심과 클러스터에 속한 샘플 사이의 거리의 제곱 합
    - 사이킷런은 **km.inertia_** 옵션 제공함
    - 이너셔 그래프에서 감소하는 속도가 꺽이는 지점을 보통 k로 지정함--> 이 지점부터 클러스터 개수를 늘려도 클러스터에 잘 밀집된 정도가 크게 개선되지 않는다.

In [ ]:
from sklearn.metrics import silhouette_score

inertia = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init='auto', random_state=42)
    km.fit(fruits_2d)
    inertia.append(km.inertia_)

    # 실루엣계수에 따른 점수
    labels = km.fit_predict(fruits_2d)
    score = silhouette_score(fruits_2d, labels)
    print(f"K={k}, Silhouette Score={score:.3f}")

plt.plot(range(2, 7), inertia)
plt.xlabel('k')
plt.ylabel('inertia')
plt.show()


### **주성분 분석(PCA)을 이용한 2D 시각화**
차원 축소하여 Clustering 시각화

In [ ]:
from sklearn.decomposition import PCA

# 고차원(10000) 데이터를 2차원으로 축소하기 위해 PCA 모델을 생성
pca = PCA(n_components=2)
fruits_pca = pca.fit_transform(fruits_2d)

# 축소된 데이터를 산점도로 시각화
optimal_k = 3
plt.figure(figsize=(8, 6))
colors = ['red', 'green', 'blue']
for i in range(optimal_k):
    cluster_data = fruits_pca[km.labels_ == i]
    plt.scatter(cluster_data[:, 0], cluster_data[:, 1], c=colors[i], label=f'Cluster {i}')

plt.title('PCA Visualization of Fruit Clusters')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()
plt.grid(True)
plt.show()

## **2. DBSCAN**

### **[예제]: DBSCAN 실습하기**
centers = [[1, 1], [-1, -1], [1, -1]]를 갖는 임의의 실수데이터 750개를 만들어 DBSCAN 실습하기

**1. 샘플 데이터 만들기**
- **make_blobs**: '덩어리(blob)' 형태의 가상 데이터를 만들어주는 함수
- 주로 클러스터링(Clustering, 군집화) 알고리즘이나 분류(Classification) 알고리즘을 테스트하고 시각적으로 이해하는 데 사용되는, 일종의 '연습용 데이터 세트 제조기'

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

# 실습용 데이터 생성하기
centers = [[1, 1], [-1, -1], [1, -1]]
X, labels_true = make_blobs(
    n_samples=750, centers=centers, cluster_std=0.4, random_state=0
)

# 데이터 정규화
# fit_transform()은 fit()과 transform()을 합한 메소드임
X = StandardScaler().fit_transform(X)

# 데이터 시각화하기
plt.scatter(X[:, 0], X[:, 1])
plt.show()

**2. DBSCAN으로 Clustering하기**

In [ ]:
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn import metrics

# DBSCAN 모델을 생성하고 데이터 X에 대해 학습(fit)을 수행함
# eps=0.3 : 하나의 점을 기준으로 반경 0.3 안에 있는 다른 점을 이웃으로 판단
# min_samples=10 : 핵심 포인트(Core Point)가 되기 위해 필요한 최소 이웃 개수 (자기 자신 포함)
db = DBSCAN(eps=0.3, min_samples=10).fit(X)

# 학습 결과, 각 데이터 포인트에 할당된 클러스터 레이블을 가져옴 (노이즈는 -1로 레이블링됨)
labels = db.labels_


# 노이즈(-1)를 제외한 실제 클러스터의 개수를 계산함
# set(labels)를 통해 유일한 레이블 값을 구하고, -1이 있으면 1을 빼서 클러스터 개수만 남김
n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)

# Noisy samples 개수 확인하기
n_noise_ = list(labels).count(-1)

print("Estimated number of clusters: %d" % n_clusters_)
print("Estimated number of noise points: %d" % n_noise_)

In [ ]:
# 평가 지표 출력하기
print(f"Homogeneity: {metrics.homogeneity_score(labels_true, labels):.3f}")
print(f"Completeness: {metrics.completeness_score(labels_true, labels):.3f}")
print(f"V-measure: {metrics.v_measure_score(labels_true, labels):.3f}")
print(f"Rand Index: {metrics.adjusted_rand_score(labels_true, labels):.3f}")
print(f"AMI: {metrics.adjusted_mutual_info_score(labels_true, labels):.3f}")
print(f"Silhouette Coefficient: {metrics.silhouette_score(X, labels):.3f}")

**3. 시각화하기**

In [ ]:
# 레이블 유일값 설정하기
unique_labels = set(labels)

# 핵심 포인트(Core Point)를 구분하기 위한 마스크(mask)를 생성함
# 우선 모든 값을 False로 채운, labels와 동일한 크기의 boolean 배열을 만듦
core_samples_mask = np.zeros_like(labels, dtype=bool)

# DBSCAN 결과에서 핵심 포인트로 식별된 샘플들의 인덱스를 가져와, 해당 위치의 값을 True로 변경함
core_samples_mask[db.core_sample_indices_] = True


# 클러스터 구분 색상 설정하기
colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]

# 각 유일한 레이블(k)과 해당 색상(col)을 하나씩 순회하며 점을 시각화하기
for k, col in zip(unique_labels, colors):
    if k == -1:
        # 레이블이 -1로 지정된 Noise Point는 검은색으로 지정하기
        col = [0, 0, 0, 1]

    # 현재 순회 중인 클러스터(k)에 속하는 데이터만 True로 표시하는 마스크를 생성함
    class_member_mask = labels == k

    # [핵심 포인트 그리기]
    # 현재 클러스터에 속하면서(&) 핵심 포인트인 데이터의 좌표(xy)를 선택함
    xy = X[class_member_mask & core_samples_mask]
    plt.plot(
        xy[:, 0],
        xy[:, 1],
        "o",
        markerfacecolor=tuple(col),
        markeredgecolor="k",
        markersize=14,
    )

    # [경계 포인트 그리기]
    # 현재 클러스터에 속하면서(&) 핵심 포인트가 아닌(~, 경계 포인트) 데이터의 좌표(xy)를 선택함
    xy = X[class_member_mask & ~core_samples_mask]
    plt.plot(
        xy[:, 0],
        xy[:, 1],
        "o",
        markerfacecolor=tuple(col),
        markeredgecolor="b",
        markersize=6,
    )

plt.title(f"Estimated number of clusters: {n_clusters_}")
plt.show()

### **[실습] 데이터모양에 따른 군집 알고리즘**
- 예: 링 원형 데이터 일 경우 적합

In [ ]:
# ============================================================
# DBSCAN 실습 - 링(원형) 데이터
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_circles
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# 1. 링 모양 데이터 생성
# ------------------------------------------------------------
X, y_true = make_circles(
    n_samples=1000,       # 데이터 개수
    factor=0.45,          # 안쪽 원의 크기
    noise=0.05,           # 약간의 노이즈 추가
    random_state=42
)

print("데이터 크기:", X.shape)

# ------------------------------------------------------------
# 2. 데이터 표준화
# ------------------------------------------------------------
X_scaled = StandardScaler().fit_transform(X)


# ------------------------------------------------------------
# 3. K-Means
# ------------------------------------------------------------
kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

kmeans_labels = kmeans.fit_predict(X_scaled)


# ------------------------------------------------------------
# 4. DBSCAN
# ------------------------------------------------------------
dbscan = DBSCAN(
    eps=0.20,             # 이웃으로 판단하는 거리
    min_samples=5         # 최소 이웃 데이터 수
)

dbscan_labels = dbscan.fit_predict(X_scaled)


# ------------------------------------------------------------
# 5. 결과 출력
# ------------------------------------------------------------
print("K-Means 군집:", np.unique(kmeans_labels))
print("DBSCAN 군집:", np.unique(dbscan_labels))

# DBSCAN에서 -1은 Noise
n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = np.sum(dbscan_labels == -1)

print("DBSCAN 군집 개수:", n_clusters)
print("Noise 데이터 수:", n_noise)


# ------------------------------------------------------------
# 6. 원본 데이터
# ------------------------------------------------------------
plt.figure(figsize=(6, 6))

plt.scatter(
    X_scaled[:, 0],
    X_scaled[:, 1],
    s=15
)

plt.title("Original Ring Data")
plt.xlabel("X1")
plt.ylabel("X2")
plt.axis("equal")
plt.show()


# ------------------------------------------------------------
# 7. K-Means 결과
# ------------------------------------------------------------
plt.figure(figsize=(6, 6))

plt.scatter(
    X_scaled[:, 0],
    X_scaled[:, 1],
    c=kmeans_labels,
    s=15
)

plt.scatter(
    kmeans.cluster_centers_[:, 0],
    kmeans.cluster_centers_[:, 1],
    marker="X",
    s=200
)

plt.title("K-Means")
plt.xlabel("X1")
plt.ylabel("X2")
plt.axis("equal")
plt.show()


# ------------------------------------------------------------
# 8. DBSCAN 결과
# ------------------------------------------------------------
plt.figure(figsize=(6, 6))

plt.scatter(
    X_scaled[:, 0],
    X_scaled[:, 1],
    c=dbscan_labels,
    s=15
)

plt.title("DBSCAN")
plt.xlabel("X1")
plt.ylabel("X2")
plt.axis("equal")
plt.show()

### **[실습] DBSCAN 를 사용하여 과일 사진 분류하기**

In [ ]:
# fruits_300.npy 데이터를 DBSCAN 으로 군집하기
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# fruits_2d 데이터 (이미 10000차원으로 reshape 됨)
# fruits_2d = fruits.reshape(-1, 100*100) # 이 부분은 이미 앞선 셀에서 수행됨

# 1. 데이터 스케일링
scaler = StandardScaler()
fruits_scaled = scaler.fit_transform(fruits_2d)

# 2. DBSCAN 모델 훈련
# eps와 min_samples는 데이터셋에 따라 조정이 필요합니다.
# 여기서는 임의의 값을 설정하여 시도해봅니다.
# 너무 작은 eps는 대부분의 데이터를 노이즈로 분류할 수 있고, 너무 큰 eps는 모든 데이터를 하나의 클러스터로 묶을 수 있습니다.
# min_samples는 일반적으로 데이터 차원의 2배를 사용하기도 하지만, 여기서는 이미지 데이터이므로 작은 값부터 시작합니다.
db = DBSCAN(eps=25, min_samples=5).fit(fruits_scaled)

# 3. 클러스터 레이블 확인
labels = db.labels_

# 노이즈(-1)를 제외한 실제 클러스터의 개수를 계산
n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
n_noise_ = list(labels).count(-1)

print(f"✅ Estimated number of clusters: {n_clusters_}")
print(f"✅ Estimated number of noise points: {n_noise_}")
print("✅ 클러스터 레이블별 샘플 수:")
unique_labels, counts = np.unique(labels, return_counts=True)
for label, count in zip(unique_labels, counts):
    if label == -1:
        print(f"  Noise: {count}")
    else:
        print(f"  Cluster {label}: {count}")

# 4. 클러스터링 결과 시각화 (draw_fruits 함수 재사용)
# 이전에 정의된 draw_fruits 함수를 사용하기 위해 다시 정의하거나, 해당 셀을 실행해야 합니다.
# 여기서는 코드가 한 셀에 독립적으로 작동하도록 다시 정의합니다.
def draw_fruits(arr, ratio=1):
    n = len(arr)
    rows = int(np.ceil(n/10))
    cols = n if rows < 2 else 10
    fig, axs = plt.subplots(rows, cols,
                            figsize=(cols*ratio, rows*ratio), squeeze=False)
    for i in range(rows):
        for j in range(cols):
            if i*10 + j < n:
                axs[i, j].imshow(arr[i*10 + j], cmap='gray_r')
            axs[i, j].axis('off')
    plt.show()

print("\n✅ DBSCAN 클러스터별 과일 이미지:")
for i in range(n_clusters_):
    print(f"--- Cluster {i} ---")
    draw_fruits(fruits[labels == i])

if n_noise_ > 0:
    print("--- Noise Points ---")
    draw_fruits(fruits[labels == -1])

# PCA를 이용한 2D 시각화
# 고차원(10000) 데이터를 2차원으로 축소하기 위해 PCA 모델을 생성
pca = PCA(n_components=2)
fruits_pca_dbscan = pca.fit_transform(fruits_scaled)

plt.figure(figsize=(10, 8))
colors_map = plt.cm.get_cmap('viridis', n_clusters_ + 1) # +1 for noise if present

for i in unique_labels:
    if i == -1:
        color = 'black' # Noise points
        label_text = 'Noise'
    else:
        color = colors_map(i)
        label_text = f'Cluster {i}'

    cluster_data = fruits_pca_dbscan[labels == i]
    plt.scatter(cluster_data[:, 0], cluster_data[:, 1], color=color, label=label_text, alpha=0.6)

plt.title('DBSCAN Clustering with PCA Visualization')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()
plt.grid(True)
plt.show()

- **1. 군집에 실패한 이유** (왜 1개의 클러스터와 290개의 노이즈가 나왔을까?)
    - **차원의 저주**: 이미지 데이터는 10,000차원(100x100)의 초고차원 데이터입니다. 고차원 공간에서는 모든 데이터 포인트 사이의 거리가 거의 일정하게 멀어져 밀도를 측정하기가 매우 어렵습니다.
    - **잘못된 스케일링의 영향**: StandardScaler는 각 픽셀별로 평균을 0, 분산을 1로 맞춥니다. 이미지 데이터에서 값의 차이가 없는 배경(가장자리 주변부 등)의 미세한 노이즈 픽셀까지 표준화 과정을 거치며 값이 증폭되어, 이미지 본연의 형태 정보를 왜곡하고 군집 성능을 떨어뜨리는 주 원인이 됩니다.
- **2. 해결 방법**
    - 이미지 데이터의 차원을 축소하여 핵심 특징만 남기는 PCA(주성분 분석)를 먼저 수행한 뒤 DBSCAN을 적용해야 합니다. PCA로 노이즈를 제거하고 주요 성분(예: 2차원 또는 10차원)만 추출하여 DBSCAN을 실행하면 더 효과적으로 군집화됩니다

#### **2-1. 성능 개선: PCA(차원 축소) 후 DBSCAN 적용하기**
- 초고차원 이미지 데이터에 직접 DBSCAN을 적용하면 거리 측정의 왜곡으로 인해 정상적인 군집화가 어렵다.
- 주성분 분석(PCA)을 통해 이미지의 핵심 특징 정보(예: 2개의 주성분)만 추출한 후, DBSCAN을 수행하여 과일들을 다시 분류

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA

# 1. 원본 데이터(fruits_2d)에서 StandardScaler를 쓰지 않고,
# 이미지 데이터에 더 적합한 단순 정규화(0~1 사이 값)를 수행하거나 PCA를 직접 적용합니다.
fruits_normalized = fruits_2d / 255.0

# 2. PCA를 사용하여 10,000차원을 2차원(PC1, PC2)으로 축소
pca = PCA(n_components=2, random_state=42)
fruits_pca_2d = pca.fit_transform(fruits_normalized)

# 3. 2차원으로 축소된 데이터 공간에서 DBSCAN 실행
# 데이터 분포가 좁아졌으므로 eps(반경) 값을 그에 맞게 조정합니다.
db_improved = DBSCAN(eps=0.5, min_samples=5).fit(fruits_pca_2d)
labels_improved = db_improved.labels_

# 결과 분석
n_clusters_imp = len(set(labels_improved)) - (1 if -1 in labels_improved else 0)
n_noise_imp = list(labels_improved).count(-1)

print(f"✅ [개선 완료] Estimated number of clusters: {n_clusters_imp}")
print(f"✅ [개선 완료] Estimated number of noise points: {n_noise_imp}")
print("✅ 클러스터별 샘플 수:")
unique_lbls, counts_lbls = np.unique(labels_improved, return_counts=True)
for label, count in zip(unique_lbls, counts_lbls):
    if label == -1:
        print(f"  Noise: {count}개")
    else:
        print(f"  Cluster {label}: {count}개")

# 4. 개선된 DBSCAN 클러스터별 과일 이미지 시각화
print("\n✅ [시각화] 개선된 DBSCAN 군집화 결과:")
for i in range(n_clusters_imp):
    print(f"--- Cluster {i} ---")
    draw_fruits(fruits[labels_improved == i])

if n_noise_imp > 0:
    print("--- Noise Points ---")
    draw_fruits(fruits[labels_improved == -1])

# 5. 2D Scatter Plot으로 결과 영역 확인
plt.figure(figsize=(10, 8))
unique_labels_set = set(labels_improved)
colors_spectral = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels_set))]

for k, col in zip(unique_labels_set, colors_spectral):
    if k == -1:
        col = [0, 0, 0, 1]  # 노이즈는 검은색

    class_member_mask = (labels_improved == k)
    xy = fruits_pca_2d[class_member_mask]
    plt.scatter(xy[:, 0], xy[:, 1], color=tuple(col), label=f'Cluster {k}' if k != -1 else 'Noise', alpha=0.7, edgecolors='k')

plt.title('Improved DBSCAN Clustering (After PCA)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()
plt.grid(True)
plt.show()

### [실습] HDBSCAN 를 사용하여 과일 사진 분류하기
** DBSCAN 성능 개선 -> PCA(차원 축소) + HDBSCAN 적용하기**
- **HDBSCAN (Hierarchical DBSCAN)** :
    - DBSCAN 을 계층적으로 확장
    - eps 를 고르지 않는다 — 모든 eps 의 군집 계층에서 안정적인 군집을 고른다
    - 밀도가 다른 군집도 찾고, 어느 군집에도 속하지 않는 점은 노이즈(−1)로 둔다

- 하이퍼파라미터
    - min_cluster_size : 군집으로 인정할 최소 데이터 개수 (기본값 5)
    - min_samples : 핵심 거리 계산에 쓰는 이웃 수 (기본값은 min_cluster_size)
결과 속성
    - labels_ : 군집 번호 (노이즈 −1)  ·  probabilities_ : 군집 소속 확률


In [ ]:
# ============================================================
# fruits_300.npy
# PCA + HDBSCAN 비지도 군집화
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import HDBSCAN

# ------------------------------------------------------------
# 1. 데이터 불러오기
# ------------------------------------------------------------
fruits = np.load("fruits_300.npy")
print("원본 데이터 shape :", fruits.shape)  # (300, 100, 100)


# ------------------------------------------------------------
# 2. 이미지 확인
# ------------------------------------------------------------
plt.figure(figsize=(10, 3))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow( fruits[i], cmap="gray" )
    plt.axis("off")

plt.suptitle("Sample Fruit Images")
plt.show()


# ------------------------------------------------------------
# 3. 100×100 이미지를 10,000차원 벡터로 변환
# ------------------------------------------------------------
X = fruits.reshape(fruits.shape[0],-1)
print("변환 후 shape :", X.shape)     # (300, 10000)


# ------------------------------------------------------------
# 4. Standard Scaling
# ------------------------------------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# ------------------------------------------------------------
# 5. PCA로 차원 축소
# ------------------------------------------------------------
# 10,000차원 → 20차원
#
# 군집화 자체를 2차원 PCA에서 하면
# 정보 손실이 너무 클 수 있으므로
# 20차원 정도를 사용
pca = PCA(
    n_components=20,
    random_state=42
)

X_pca = pca.fit_transform(X_scaled)
print("PCA 변환 shape :", X_pca.shape)
print(
    "설명된 분산 비율:",
    pca.explained_variance_ratio_.sum()
)


# ------------------------------------------------------------
# 6. HDBSCAN
# ------------------------------------------------------------
model = HDBSCAN(
    min_cluster_size=10,
    min_samples=5
)
labels = model.fit_predict(X_pca)


# ------------------------------------------------------------
# 7. 군집 결과 확인
# ------------------------------------------------------------
print("\nCluster Label")
print(np.unique(labels))

# -1 : Noise
clusters = set(labels)
n_clusters = len(
    clusters - {-1}
)
n_noise = np.sum(labels == -1)
print("Cluster 개수 :", n_clusters)
print("Noise 개수 :", n_noise)


# ------------------------------------------------------------
# 8. 시각화를 위해 PCA 2차원
# ------------------------------------------------------------
pca2 = PCA(
    n_components=2,
    random_state=42
)
X_2d = pca2.fit_transform(X_scaled)


# ------------------------------------------------------------
# 9. 시각화 (산점도)
# ------------------------------------------------------------
plt.figure(figsize=(10, 7))
for label in np.unique(labels):
    mask = labels == label
    if label == -1:
        plt.scatter(
            X_2d[mask, 0],
            X_2d[mask, 1],
            s=35,
            alpha=0.5,
            label="Noise"
        )

    else:
        plt.scatter(
            X_2d[mask, 0],
            X_2d[mask, 1],
            s=40,
            label=f"Cluster {label}"
        )


plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("HDBSCAN Clustering with PCA Visualization")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


# ------------------------------------------------------------
# 10. 각 클러스터 및 노이즈에 속한 실제 과일 이미지 시각화
# ------------------------------------------------------------
def draw_fruits(arr, ratio=1):
    n = len(arr)
    if n == 0:
        return
    rows = int(np.ceil(n/10))
    cols = n if rows < 2 else 10
    fig, axs = plt.subplots(rows, cols,
                            figsize=(cols*ratio, rows*ratio), squeeze=False)
    for i in range(rows):
        for j in range(cols):
            if i*10 + j < n:
                axs[i, j].imshow(arr[i*10 + j], cmap='gray_r')
            axs[i, j].axis('off')
    plt.show()

print("\n============================================================")
print("✅ HDBSCAN 클러스터별 상세 이미지 시각화")
print("============================================================")

for i in sorted(list(clusters - {-1})):
    cluster_fruits = fruits[labels == i]
    print(f"\n--- Cluster {i} (총 {len(cluster_fruits)}개) ---")
    draw_fruits(cluster_fruits)

if n_noise > 0:
    noise_fruits = fruits[labels == -1]
    print(f"\n--- Noise Points (총 {len(noise_fruits)}개) ---")
    draw_fruits(noise_fruits)

---

## **[정리] "만능 군집 알고리즘은 없다"**

### **Clustering 알고리즘 5종 × 데이터 모양 4종**

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_moons, make_circles
from sklearn.cluster import KMeans, DBSCAN, HDBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

rs = np.random.RandomState(SEED)
Xb, yb = make_blobs(500, centers=3, random_state=8)
Xa = Xb @ np.array([[0.6, -0.6], [-0.4, 0.8]])                  # 길쭉한(비등방) 군집
Xv, yv = make_blobs(500, centers=3, cluster_std=[1.0, 2.5, 0.5], random_state=170)

datasets = {"둥근 덩어리": (Xb, yb), "길쭉한 덩어리": (Xa, yb), "초승달": make_moons(500, noise=.06, random_state=SEED),
            "밀도 제각각": (Xv, yv)}

algos = {"K-means": lambda: KMeans(3, n_init=10, random_state=SEED),
         "GMM": lambda: GaussianMixture(3, covariance_type="full", random_state=SEED),
         "Agglomerative(ward)": lambda: AgglomerativeClustering(3),
         "DBSCAN": lambda: DBSCAN(eps=0.25, min_samples=8),
         "HDBSCAN": lambda: HDBSCAN(min_cluster_size=20)}

fig, axes = plt.subplots(len(datasets), len(algos), figsize=(15, 11))

ari = pd.DataFrame(index=list(datasets), columns=list(algos), dtype=float)

for i, (dname, (Xd, yd)) in enumerate(datasets.items()):
    Xs = StandardScaler().fit_transform(Xd)
    for j, (aname, make) in enumerate(algos.items()):
        m = make()
        lab = m.fit_predict(Xs)
        ari.loc[dname, aname] = adjusted_rand_score(yd, lab)
        axes[i, j].scatter(Xs[:, 0], Xs[:, 1], c=lab, s=6, cmap="tab10")
        axes[i, j].set_title(f"{aname}\nARI={ari.loc[dname, aname]:.2f}", fontsize=9)
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
    axes[i, 0].set_ylabel(dname)

plt.tight_layout(); plt.show()
print(ari.round(2).to_string())



---



## **[미션] 고객 세분화 모델 구현 실습하기**

### **1) 데이터 로드 및 전처리하기**

In [ ]:
# openpyxl 설치하기
# !pip install openpyxl

In [ ]:
# 필요한 라이브러리 불러오기
import datetime
import numpy as np
import datetime as dt
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 경고 메시지를 무시하도록 설정하기
import warnings
warnings.filterwarnings("ignore")

# 엑셀파일에서 데이터를 로드해서 데이터프레임으로 저장하기
df = pd.read_excel(
    'https://archive.ics.uci.edu/ml/machine-learning-databases/00502/online_retail_II.xlsx',
    engine="openpyxl")

# 데이터프레임의 처음 5행의 데이터 출력하기
df.head(5)

In [ ]:
# 데이터셋 정보 확인하기
df.info()
df.describe()

In [ ]:
# 결측치의 합계를 구해서 내림차순으로 정렬하기
df.isnull().sum().sort_values(ascending=False)

# 제품 수량과 가격이 0 이하인 데이터를 제거하기
df = df[(df['Quantity']>0) & (df['Price']>0)]

# 중복 데이터를 제거하기
df = df.drop_duplicates()

### **2) RFM 분석 기법을 이용한 고객 분류하기**
- 고객관리(CRM, Customer Relation Management) 기법으로 고객 세분화를 위해
    - **Recency(구매 최신성)**:  얼마나 최근에 구매했는가?
        - 마지막 구매 날짜의 다음날
        - last_date = df.InvoiceDate.max() +  datetime.timedelta(days=1)
    - **Frequency(구매 빈도)**: 얼마나 자주 구매했는가?
    - **Monetary(구매 금액)**: 얼마나 많은 금액을 지출했는가?  RFM 측면에서 데이터를 확인해 본다.
        - 구매금액 = 구매수량x제품가격
        - df['Amount'] = df['Quantity'] * df['Price']

- **RFM 지표 계산하기**

In [ ]:
# 최신성(Recency) = 마지막 구매 날짜의 다음날
last_date = df.InvoiceDate.max() +  datetime.timedelta(days=1)

# 구매금액 = 구매수량 x 제품가격
df['Amount'] = df['Quantity'] * df['Price']

# 구매빈도 = groupby 함수로 고객별로 구매건수
rfm = df.groupby('Customer ID').agg(
{'InvoiceDate': lambda InvoiceDate: (last_date - InvoiceDate.max()).days,
 'Invoice': lambda Invoice: Invoice.nunique(),
 'Amount': lambda Amount: Amount.sum()})


# 데이터프레임의 칼럼명 변경하기
rfm.columns = ['recency', 'frequency', 'monetary']
rfm.head()

- **고객 점수 계산하기**

In [ ]:
# 최신성, 구매빈도, 구매금액을 1에서 5까지의 점수로 환산하기
rfm['recency_score'] = pd.qcut(rfm['recency'], 5,
                            labels=[5, 4, 3, 2, 1]).astype(int)
rfm['frequency_score'] = pd.qcut(rfm['frequency'].rank(method="first"), 5,
                            labels=[1, 2, 3, 4, 5]).astype(int)
rfm['monetary_score'] = pd.qcut(rfm['monetary'], 5,
                            labels=[1, 2, 3, 4, 5]).astype(int)

# 최신성, 구매빈도, 구매금액의 환산점수 합계를 고객 점수(customer_score)로 저장하기
rfm['customer_score'] = rfm['recency_score'] + \
                        rfm['frequency_score'] + rfm['monetary_score']

# 데이터 확인하기
rfm.head()

# 고객 점수(customer_score)가 만점(15점)인 고객 ID 확인하기
rfm[rfm['customer_score']==15].sort_values(
    'monetary', ascending=False).head()

In [ ]:
# 고객 등급 분류 함수 만들기
def level(score):
    if  score > 12 :
        return 'VIP'
    elif 9 < score <= 12:
        return 'GOLD'
    elif 5 < score <= 9 :
        return 'SILVER'
    else:
        return 'WHITE'

# 고객 등급 분류 함수를 적용하여 고객 등급(level) 데이터 생성하기
rfm['level'] = rfm['customer_score'].apply(
    lambda customer_score : level(customer_score))
rfm.head()

# 고객 등급별 고객 수 그래프 출력하기
sns.countplot(x=rfm['level'])

### **3) K-평균 군집화 알고리즘을 이용한 고객 분류**(Customer Segmentation)

In [ ]:
# 데이터프레임의 요약 통계량 확인하기
rfm.describe()

- **이상치(outlier) 제거하기**

In [ ]:
#이상치 제거 함수 작성하기
def processing_outlier(df, col_nm):
    Q1 = df[col_nm].quantile(0.25)
    Q3 = df[col_nm].quantile(0.75)
    IQR = Q3 - Q1
    df = df[(df[col_nm] >= Q1 - 1.5*IQR) & (df[col_nm] <= Q3 + 1.5*IQR)]
    return df

# 비교를 위해 이상치 제거 전 데이터 저장하기
rfm_tmp = rfm.copy()


# recency 이상치 제거하기
rfm = processing_outlier(rfm, 'recency')

# 이상치 제거 전후 비교하기
fig = plt.figure(figsize=(14, 5)) # 그림 사이즈 지정 (가로 14인치, 세로 5인치)
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2)
plt.title('Recency')

# 이상치 제거 전 그래프 시각화
ax1.boxplot(rfm_tmp.recency)

# 이상치 제거 후 그래프 시각화
ax2.boxplot(rfm.recency)
plt.show()


In [ ]:
# monetary 이상치 제거하기
rfm = processing_outlier(rfm, 'monetary')

# 이상치 제거 전후 비교하기
fig = plt.figure(figsize=(14, 5)) # 그림 사이즈 지정 (가로 14인치, 세로 5인치)
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2)
plt.title('Monetary')

# 이상치 제거 전 그래프 시각화
ax1.boxplot(rfm_tmp.monetary)

# 이상치 제거 후 그래프 시각화
ax2.boxplot(rfm.monetary)
plt.show()

In [ ]:
rfm_k = rfm[['recency','frequency','monetary']]

# 데이터 표준화하기
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_k)

print(rfm_scaled)



ks = range(1,11)
inertias=[]
for k in ks :
    kc = KMeans(n_clusters=k,random_state=42)
    kc.fit(rfm_scaled)
    cluster = kc.fit_predict(rfm_scaled)
    inertias.append(kc.inertia_)

# k vs inertia 그래프 그리기
plt.subplots(figsize=(10, 6))
plt.plot(ks, inertias, "bo-")
plt.xlabel("$k$", fontsize=14)
plt.ylabel('Inertia')
plt.annotate('Elbow',
             xy=(3, inertias[2]),
             xytext=(0.55, 0.55),
             textcoords='figure fraction',
             fontsize=16,
             arrowprops=dict(facecolor='black', shrink=0.1)
            )
plt.xticks(ks)
plt.style.use('ggplot')
plt.title('Best Number for KMeans')
plt.show()

In [ ]:
# K를 3으로 설정하고 군집 중심 찾기
kc = KMeans(3,random_state=42)
kc.fit(rfm_scaled)
identified_clusters = kc.fit_predict(rfm_k)
clusters_scaled = rfm_k.copy()
clusters_scaled['cluster_pred'] = kc.fit_predict(rfm_scaled)

print(f'Identified Clusters : {identified_clusters}')
print(f'Cluster Centers :\n{kc.cluster_centers_}')

In [ ]:
# cluster_pred 칼럼의 값별로 count 하기
f, ax = plt.subplots(figsize=(25, 5))
ax = sns.countplot(x='cluster_pred', data=clusters_scaled)

# Pandas groupby 함수로 cluster_pred별 개수 집계하기
print(clusters_scaled.groupby(['cluster_pred']).count())


rfm_k['cluster'] = clusters_scaled['cluster_pred']
rfm_k['level'] = rfm['level']

# recency, frequency, monetary의 평균값, 최솟값, 최댓값 구하기
rfm_k.groupby('cluster').agg({
    'recency' : ['mean','min','max'],
    'frequency' : ['mean','min','max'],
    'monetary' : ['mean','min','max','count']
})

### **4) AIHub 데이터: 카드 고객 RFM 세분화** → 군집 프로파일 → 마케팅 액션
- **R**ecency(최근성) · **F**requency(빈도) · **M**onetary(금액) — 1990년대부터 쓰인 고객관리(CRM) 기법
- **한국 카드거래 합성데이터**로 수행 (카드KEY = 고객)

In [ ]:
# 데이터 가져오기
def first_existing(*paths):
    return next((p for p in paths if os.path.exists(p)), paths[0])

CARD_ZIP = first_existing("VL_카드거래.zip", "VL_카드거래.zip")

In [ ]:
def read_card_zip(zip_path):
    zf = zipfile.ZipFile(zip_path)
    frames = []
    for n in sorted(zf.namelist()):
        if n.endswith(".csv"):
            with zf.open(n) as f:
                frames.append(pd.read_csv(f, dtype=str, encoding="utf-8-sig", na_values=["_", ""]))
    d = pd.concat(frames, ignore_index=True)
    d["승인일자"] = pd.to_datetime(d["승인일자"], format="%Y%m%d")
    for c in ["통합승인금액", "카드이용한도금액", "승인시간대", "연령", "경과일수_최종이용일자", "전월_매출건수", "전월_매출금액", "할부가능개월수"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d["y"] = (d["이상거래여부"] == "1").astype(int)
    return d


card = read_card_zip(CARD_ZIP)
print("카드거래:", card.shape, "| 기간", card["승인일자"].min().date(), "~", card["승인일자"].max().date(),
      "| 고객(카드KEY)", card["카드KEY"].nunique(), "| 이상거래 %.2f%%" % (100 * card.y.mean()))

In [ ]:
snap = card["승인일자"].max() + pd.Timedelta(days=1)                   # 기준일 = 마지막 거래 다음날
rfm = card.groupby("카드KEY").agg(recency=("승인일자", lambda s: (snap - s.max()).days),
                                  frequency=("승인SEQ", "size"),
                                  monetary=("통합승인금액", "sum"),
                                  fraud_rate=("y", "mean"))           # 이상거래 비율(군집 해석용 — 군집 학습에는 안 씀)
print(rfm.describe().round(1).to_string())


# 오른쪽으로 긴 꼬리 → 이상치 '삭제' 대신 log1p 로 압축 (작년 IQR 삭제 방식 개선: 고객을 버리지 않는다)
Z = StandardScaler().fit_transform(np.log1p(rfm[["recency", "frequency", "monetary"]]))
sil = {k: silhouette_score(Z, KMeans(k, n_init=10, random_state=SEED).fit_predict(Z)) for k in range(2, 9)}
print("k별 실루엣:", {k: round(v, 3) for k, v in sil.items()})

# 실루엣 점수가 가장 높은 K 값 선택
# K = max(sil, key=sil.get)    # 최적의 실루엣 점수를 갖는 k 선택
K = 4
rfm["cluster"] = KMeans(K, n_init=10, random_state=SEED).fit_predict(Z)
profile = rfm.groupby("cluster").agg(고객수=("recency", "size"), R_중앙값=("recency", "median"),
                                     F_중앙값=("frequency", "median"), M_중앙값=("monetary", "median"),
                                     이상거래비율=("fraud_rate", "mean")).round(3)
print(profile.to_string())

In [ ]:
# 군집에 '이름'과 '행동'을 붙인다 — 규칙: R 낮을수록 최근, F·M 높을수록 우량
def persona(r):
    recent, heavy = r.R_중앙값 <= profile.R_중앙값.median(), r.M_중앙값 >= profile.M_중앙값.median()
    return {(True, True): "핵심 우량 고객 → 혜택 유지·VIP", (True, False): "최근 가벼운 고객 → 교차 판매",
            (False, True): "이탈 위험 우량 → 재활성 쿠폰", (False, False): "휴면 고객 → 저비용 리마인드"}[(recent, heavy)]
profile["페르소나 → 액션(예시)"] = profile.apply(persona, axis=1)
print(profile.to_string())


# 2D 산점도 그리기 (예: Recency vs Monetary)
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111) # 2D 플롯

# 각 군집별 색상 및 마커 설정은 3D 플롯과 동일하게 재사용
colors = ['r', 'g', 'b', 'c', 'm', 'y', 'k', 'purple']
markers = ['o', '^', 's', 'D', 'P', '*', 'h', 'X']

# 각 군집 데이터를 플로팅
for i, cluster_id in enumerate(profile.index):
    # 해당 군집에 속하는 고객 데이터 필터링
    cluster_data = rfm[rfm['cluster'] == cluster_id]

    ax.scatter(
        cluster_data['recency'],
        cluster_data['monetary'],
        color=colors[i % len(colors)],
        marker=markers[i % len(markers)],
        s=10, # 마커 크기는 동일하게 유지하거나 필요에 따라 조정
        alpha=0.6, # 투명도 설정
        label=f'군집 {cluster_id}: {profile.loc[cluster_id, "페르소나 → 액션(예시)"]}'
    )

# 축 레이블 설정
ax.set_xlabel('Recency (최신성)')
ax.set_ylabel('Monetary (구매 금액)')
ax.set_title('RFM Clustering Results - 2D Scatter Plot (Recency vs Monetary)')

# 범례 표시
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
# 군집결과 시각화 코드
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 3D 산점도 그리기
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# 각 군집별 색상 및 마커 설정
colors = ['r', 'g', 'b', 'c', 'm', 'y', 'k', 'purple'] # 최대 8개 군집까지 지원
markers = ['o', '^', 's', 'D', 'P', '*', 'h', 'X']

# 각 군집 데이터를 플로팅
for i, cluster_id in enumerate(profile.index):
    ax.scatter(
        profile.loc[cluster_id, 'R_중앙값'],
        profile.loc[cluster_id, 'F_중앙값'],
        profile.loc[cluster_id, 'M_중앙값'],
        color=colors[i % len(colors)],
        marker=markers[i % len(markers)],
        s=profile.loc[cluster_id, '고객수'] / 5, # 고객 수에 비례하여 마커 크기 조정
        label=f'군집 {cluster_id}: {profile.loc[cluster_id, "페르소나 → 액션(예시)"]}'
    )

# 축 레이블 설정
ax.set_xlabel('Recency (R_중앙값)')
ax.set_ylabel('Frequency (F_중앙값)')
ax.set_zlabel('Monetary (M_중앙값)')
ax.set_title('RFM Clustering Results - 3D Scatter Plot')

# 범례 표시
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()





---



### **[LLM 실습] 문서 임베딩 + K-means로 주제 자동 분류**
텍스트를 벡터로 만들어 군집화한다.
GPU 모델 선택해서 실행

In [ ]:
!pip install -q sentence-transformers scikit-learn pandas

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd

docs = [
    "트랜스포머는 어텐션 기반 구조다",
    "GPU 가격이 계속 오르고 있다",
    "셀프 어텐션은 병렬 처리가 가능하다",
    "반도체 수출이 증가했다",
    "손흥민이 결승골을 넣었다",
    "메모리 반도체 시장이 회복 중이다",
    "야구 한국시리즈가 시작됐다",
    "BERT는 인코더만 사용한다",
    "프로축구 관중이 늘었다"
]

X = SentenceTransformer("BAAI/bge-m3").encode(
    docs,
    normalize_embeddings=True
)

max_silhouette_score = -1
best_k = -1

for k in range(2, 7):
    km = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=0
    ).fit(X)

    current_silhouette_score = silhouette_score(X, km.labels_)
    print(
        f"k={k}  실루엣="
        f"{current_silhouette_score:.3f}"
    )

    if current_silhouette_score > max_silhouette_score:
        max_silhouette_score = current_silhouette_score
        best_k = k


print(f"\n최적의 k 값: {best_k} (실루엣 점수: {max_silhouette_score:.3f})")

best = KMeans(
    n_clusters=best_k, # 최적의 k 값을 사용하여 KMeans 모델 초기화
    n_init=10,
    random_state=0
).fit(X)

print(
    pd.DataFrame({
        "문서": docs,
        "군집": best.labels_
    })
    .sort_values("군집")
    .to_string()
)


#### 3-1. 군집에 이름을 붙이는 작업을 LLM에게 시켜보기 (군집별 문서 → 주제명)

In [ ]:
import numpy as np
import google.generativeai as genai
from google.colab import userdata

# Gemini API 클라이언트 초기화 (genai.configure 사용)
genai.configure(
    api_key=userdata.get("GEMINI_API_KEY")
)

TEMPERATURE = 0.2

print("\n--- LLM을 이용한 군집별 주제명 부여 ---")

# 군집별 문서들을 저장할 딕셔너리 생성
clustered_docs = {i: [] for i in np.unique(best.labels_)}
for doc, label in zip(docs, best.labels_):
    clustered_docs[label].append(doc)

# 모델 로드
model = genai.GenerativeModel('gemini-3.5-flash-lite') # 'gemini-pro' 모델을 사용합니다.

# 각 군집에 대해 LLM에게 주제명 요청
for cluster_id, cluster_doc_list in clustered_docs.items():
    if not cluster_doc_list:
        continue

    # LLM 프롬프트 생성
    prompt = f"""
다음 문서들은 하나의 군집을 이룹니다. 이 문서들이 공통적으로 다루는 주제를 3단어 이내로 명확하게 요약해주세요. 추가 설명은 하지 말고 주제명만 제시해주세요.

문서 목록:
{chr(10).join(f'- {d}' for d in cluster_doc_list)}
"""

    print(f"\n--- 군집 {cluster_id} 프롬프트 ---")
    print(prompt)

    # LLM 호출
    try:
        # genai.GenerativeModel.generate_content 사용
        response = model.generate_content(prompt, generation_config={
            "temperature": TEMPERATURE
        })

        topic_name = response.text.strip()
        print(f"군집 {cluster_id} 주제: {topic_name}")
    except Exception as e:
        print(f"군집 {cluster_id} 주제를 생성하는 중 오류 발생: {e}")
        print(f"문서 목록: {cluster_doc_list}")

